# Setup (Imports + Seed + GPU check)

In [ ]:
import os, random, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

def cleanup():
    tf.keras.backend.clear_session()
    gc.collect()

TF version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Load + Normalize Fashion-MNIST + Train/Val split

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
from sklearn.model_selection import train_test_split

CLASS_NAMES = [
    "T-shirt/top","Trouser","Pullover","Dress","Coat",
    "Sandal","Shirt","Sneaker","Bag","Ankle boot"
]

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

# CNN versions need channel dim
x_tr_cnn  = x_tr[..., None]
x_val_cnn = x_val[..., None]
x_test_cnn = x_test[..., None]

print("Train:", x_tr.shape, "Val:", x_val.shape, "Test:", x_test.shape)

Train: (54000, 28, 28) Val: (6000, 28, 28) Test: (10000, 28, 28)


# Compile + Train + Evaluate (EarlyStopping)

In [ ]:
def compile_and_train(model, xtr, ytr, xva, yva,
                      optimizer="adam",
                      epochs=25,
                      batch_size=128,
                      verbose=0):

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=3,
            restore_best_weights=True
        )
    ]

    history = model.fit(
        xtr, ytr,
        validation_data=(xva, yva),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
        callbacks=callbacks
    )
    return history

def eval_acc(model, xva, yva, xte, yte):
    val_loss, val_acc = model.evaluate(xva, yva, verbose=0)
    test_loss, test_acc = model.evaluate(xte, yte, verbose=0)
    return float(val_acc), float(test_acc)

def params(model):
    return int(model.count_params())

# PART 2: Dense-only (MLP)

## Build MLP + Run Experiments (1 vs 2 hidden layers; units 32/64/128/256)

In [ ]:
def build_mlp(hidden_layers=1, units=128, input_shape=(28,28), num_classes=10):
    model = keras.Sequential(name=f"MLP_h{hidden_layers}_u{units}")
    model.add(layers.Input(shape=input_shape))
    model.add(layers.Flatten())
    if hidden_layers == 1:
        model.add(layers.Dense(units, activation="relu"))
    elif hidden_layers == 2:
        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.Dense(units, activation="relu"))
    else:
        raise ValueError("hidden_layers must be 1 or 2")

    model.add(layers.Dense(num_classes, activation="softmax"))
    return model

mlp_rows = []
for h in [1, 2]:
    for u in [32, 64, 128, 256]:
        set_seed(42)
        cleanup()

        model = build_mlp(hidden_layers=h, units=u)

        hist = compile_and_train(
            model, x_tr, y_tr, x_val, y_val,
            optimizer="adam",
            epochs=25,
            batch_size=128,
            verbose=0
        )

        best_val = max(hist.history["val_accuracy"])
        val_acc, test_acc = eval_acc(model, x_val, y_val, x_test, y_test)

        mlp_rows.append({
            "model": model.name,
            "hidden_layers": h,
            "units": u,
            "params": params(model),
            "best_val_acc": float(best_val),
            "val_acc": val_acc,
            "test_acc": test_acc
        })

mlp_df = pd.DataFrame(mlp_rows).sort_values(by=["test_acc","params"], ascending=[False, True]).reset_index(drop=True)
mlp_df

,model,hidden_layers,units,params,best_val_acc,val_acc,test_acc
0,MLP_h2_u128,2,128,118282,0.900500,0.900500,0.8842
1,MLP_h1_u128,1,128,101770,0.901667,0.901667,0.8797
2,MLP_h1_u256,1,256,203530,0.899500,0.899500,0.8793
3,MLP_h2_u64,2,64,55050,0.892500,0.892500,0.8757
4,MLP_h2_u256,2,256,269322,0.888667,0.888667,0.8737
5,MLP_h1_u64,1,64,50890,0.887000,0.887000,0.8713
6,MLP_h2_u32,2,32,26506,0.887833,0.887833,0.8687
7,MLP_h1_u32,1,32,25450,0.885167,0.885167,0.8667


## Pick Best MLP (competitive accuracy with fewer params)

In [ ]:
top = mlp_df["test_acc"].max()
candidates = mlp_df[mlp_df["test_acc"] >= top - 0.005].copy()
best_mlp_row = candidates.sort_values(by=["params","test_acc"], ascending=[True, False]).iloc[0]
best_mlp_row

,1
model,MLP_h1_u128
hidden_layers,1
units,128
params,101770
best_val_acc,0.901667
val_acc,0.901667
test_acc,0.8797


# PART 3: CNN Experiments

## CNN builder (3x3 conv, maxpool, 1 dense between conv and output)

In [ ]:
def build_cnn(conv_filters, dense_units=128, dropout_rate=None, input_shape=(28,28,1), num_classes=10, name="CNN"):
    model = keras.Sequential(name=name)
    model.add(layers.Input(shape=input_shape))

    for i, f in enumerate(conv_filters):
        model.add(layers.Conv2D(f, (3,3), padding="same", activation="relu"))
        if dropout_rate is not None:
            model.add(layers.Dropout(dropout_rate))
        model.add(layers.MaxPooling2D((2,2)))

    model.add(layers.Flatten())
    model.add(layers.Dense(dense_units, activation="relu"))
    if dropout_rate is not None:
        model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(num_classes, activation="softmax"))
    return model

## 3a: Compare 1 vs 2 vs 3 conv layers

	•	1 layer: [128]
	•	2 layers: [64,128]
	•	3 layers: [32,64,128]

In [ ]:
archs = {
    "conv1_128": [128],
    "conv2_64_128": [64, 128],
    "conv3_32_64_128": [32, 64, 128]
}

cnn_a_rows = []
for name, filt in archs.items():
    set_seed(42)
    cleanup()

    model = build_cnn(conv_filters=filt, dense_units=128, dropout_rate=None, name=name)

    hist = compile_and_train(
        model, x_tr_cnn, y_tr, x_val_cnn, y_val,
        optimizer="adam",
        epochs=25,
        batch_size=128,
        verbose=0
    )

    best_val = max(hist.history["val_accuracy"])
    val_acc, test_acc = eval_acc(model, x_val_cnn, y_val, x_test_cnn, y_test)

    cnn_a_rows.append({
        "arch": name,
        "filters": tuple(filt),
        "params": params(model),
        "best_val_acc": float(best_val),
        "val_acc": val_acc,
        "test_acc": test_acc
    })

cnn_a_df = pd.DataFrame(cnn_a_rows).sort_values(by=["test_acc","params"], ascending=[False, True]).reset_index(drop=True)
cnn_a_df

,arch,filters,params,best_val_acc,val_acc,test_acc
0,conv2_64_128,"[64, 128]",878730,0.923833,0.923833,0.9121
1,conv1_128,[128],3213962,0.920833,0.920833,0.9112
2,conv3_32_64_128,"[32, 64, 128]",241546,0.916167,0.916167,0.9042


Picking best architecture

In [ ]:
best_arch = cnn_a_df.iloc[0]
best_filters = list(best_arch["filters"])
best_arch, best_filters

(arch            conv2_64_128
 filters            [64, 128]
 params                878730
 best_val_acc        0.923833
 val_acc             0.923833
 test_acc              0.9121
 Name: 0, dtype: object,
 [64, 128])

## 3b Dropout sweep (0.0, 0.1, 0.3, 0.5)

Dropout inserted after each conv and after dense

In [ ]:
drop_rates = [0.0, 0.1, 0.3, 0.5]

cnn_b_rows = []
for dr in drop_rates:
    set_seed(42)
    cleanup()

    # dr=0.0 means no dropout layers (keeps model identical to base)
    if dr == 0.0:
        model = build_cnn(best_filters, dense_units=128, dropout_rate=None, name=f"BestArch_dropout0.0")
    else:
        model = build_cnn(best_filters, dense_units=128, dropout_rate=dr, name=f"BestArch_dropout{dr}")

    hist = compile_and_train(
        model, x_tr_cnn, y_tr, x_val_cnn, y_val,
        optimizer="adam",
        epochs=35,
        batch_size=128,
        verbose=0
    )

    best_val = max(hist.history["val_accuracy"])
    val_acc, test_acc = eval_acc(model, x_val_cnn, y_val, x_test_cnn, y_test)

    cnn_b_rows.append({
        "dropout": dr,
        "params": params(model),
        "best_val_acc": float(best_val),
        "val_acc": val_acc,
        "test_acc": test_acc
    })

cnn_b_df = pd.DataFrame(cnn_b_rows).sort_values(by=["test_acc","best_val_acc"], ascending=[False, False]).reset_index(drop=True)
cnn_b_df

,dropout,params,best_val_acc,val_acc,test_acc
0,0.3,878730,0.9320,0.9320,0.9213
1,0.0,878730,0.9260,0.9260,0.9192
2,0.1,878730,0.9280,0.9280,0.9187
3,0.5,878730,0.9315,0.9315,0.9182


Picking best dropout rate

In [ ]:
best_dropout = float(cnn_b_df.iloc[0]["dropout"])
best_dropout

0.3

## 3c: Optimizer comparison (adam, rmsprop, SGD)

Using best architecture + best dropout from (b).

In [ ]:
opts = ["adam", "rmsprop", "SGD"]

cnn_c_rows = []
for opt in opts:
    set_seed(42)
    cleanup()

    if best_dropout == 0.0:
        model = build_cnn(best_filters, dense_units=128, dropout_rate=None, name=f"BestCNN_{opt}")
    else:
        model = build_cnn(best_filters, dense_units=128, dropout_rate=best_dropout, name=f"BestCNN_{opt}")

    # Optional: make SGD more competitive by using momentum
    # if opt == "SGD":
    #     optimizer_obj = keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
    # else:
    #     optimizer_obj = opt
    optimizer_obj = opt

    hist = compile_and_train(
        model, x_tr_cnn, y_tr, x_val_cnn, y_val,
        optimizer=optimizer_obj,
        epochs=35,
        batch_size=128,
        verbose=0
    )

    best_val = max(hist.history["val_accuracy"])
    val_acc, test_acc = eval_acc(model, x_val_cnn, y_val, x_test_cnn, y_test)

    cnn_c_rows.append({
        "optimizer": opt,
        "params": params(model),
        "best_val_acc": float(best_val),
        "val_acc": val_acc,
        "test_acc": test_acc
    })

cnn_c_df = pd.DataFrame(cnn_c_rows).sort_values(by=["test_acc","best_val_acc"], ascending=[False, False]).reset_index(drop=True)
cnn_c_df

,optimizer,params,best_val_acc,val_acc,test_acc
0,rmsprop,878730,0.937333,0.937333,0.9262
1,adam,878730,0.933667,0.933667,0.9236
2,SGD,878730,0.888833,0.888833,0.8779


Picking best CNN overall

In [ ]:
best_cnn_row = cnn_c_df.iloc[0]
best_cnn_row

,0
optimizer,rmsprop
params,878730
best_val_acc,0.937333
val_acc,0.937333
test_acc,0.9262


# PART 4: Final Comparison (Best MLP vs Best CNN)

## Re-training best MLP and best CNN once (clean) and show final table

In [ ]:
#  Best MLP re-train
set_seed(42)
cleanup()

best_mlp_model = build_mlp(
    hidden_layers=int(best_mlp_row["hidden_layers"]),
    units=int(best_mlp_row["units"])
)

_ = compile_and_train(
    best_mlp_model, x_tr, y_tr, x_val, y_val,
    optimizer="adam",
    epochs=25,
    batch_size=128,
    verbose=0
)

mlp_val_acc, mlp_test_acc = eval_acc(best_mlp_model, x_val, y_val, x_test, y_test)

# Best CNN re-train
set_seed(42)
cleanup()

best_opt = best_cnn_row["optimizer"]

if best_dropout == 0.0:
    best_cnn_model = build_cnn(best_filters, dense_units=128, dropout_rate=None, name="BestCNN_Final")
else:
    best_cnn_model = build_cnn(best_filters, dense_units=128, dropout_rate=best_dropout, name="BestCNN_Final")

_ = compile_and_train(
    best_cnn_model, x_tr_cnn, y_tr, x_val_cnn, y_val,
    optimizer=best_opt,
    epochs=35,
    batch_size=128,
    verbose=0
)

cnn_val_acc, cnn_test_acc = eval_acc(best_cnn_model, x_val_cnn, y_val, x_test_cnn, y_test)

final_df = pd.DataFrame([
    {"model_type":"MLP", "model":best_mlp_model.name, "params":params(best_mlp_model),
     "val_acc":mlp_val_acc, "test_acc":mlp_test_acc},
    {"model_type":"CNN", "model":best_cnn_model.name, "params":params(best_cnn_model),
     "val_acc":cnn_val_acc, "test_acc":cnn_test_acc,
     "filters":str(best_filters), "dropout":best_dropout, "optimizer":best_opt}
])

final_df

,model_type,model,params,val_acc,test_acc,filters,dropout,optimizer
0,MLP,MLP_h1_u128,101770,0.901667,0.8797,NaN,NaN,NaN
1,CNN,BestCNN_Final,878730,0.932500,0.9227,"[64, 128]",0.3,rmsprop
